#### Using Pydantic base class with LLM calls

In [14]:
from dataclasses import dataclass

# if you're using Python 3.7 or later.

In [15]:
@dataclass
class Person:
	name: str
	age: int

In [16]:
Person(name="Sam", age="10")

Person(name='Sam', age='10')

By using the dataclass decorator, we can pass in the values as strings without any complaints from the dataclass. This would mean that we could run into issues later on if we try to use the age field as an int.

In [17]:
Person(name="Sam", age="10").age

'10'

In [18]:
Person(name="Sam", age="10").age + 1

TypeError: can only concatenate str (not "int") to str

However, if we use Pydantic, we will obtain the correct type!

In [19]:
from pydantic import BaseModel

In [20]:
class Person(BaseModel):
	name: str
	age: int

In [21]:
Person(name="Sam", age="10")

Person(name='Sam', age=10)

In [22]:
Person(name="Sam", age="10").age + 1

11

If we provide data that cannot be converted to an int, an error will be returned.

In [23]:
Person(name="Sam", age="13.4")

ValidationError: 1 validation error for Person
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='13.4', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/int_parsing

#### Example

In [24]:
import os
from pydantic import BaseModel
from openai import OpenAI

In [25]:
openai_api_key = os.getenv("OPENAI_API_KEY")
                           
client = OpenAI(
    #api_key = openai_api_key
)

In [26]:
class PythonPackage(BaseModel):
    name: str
    author: str

In [27]:
resp = client.chat.completions.create(
    model   = "gpt-3.5-turbo",
    messages= [
        {
            "role": "user",
            "content": "Return the `name`, `nickname` and `author` of pydantic, in a json object."
        },
    ]
)

In [28]:
resp.choices[0].message.content

'{\n    "name": "pydantic",\n    "nickname": "pydantic",\n    "author": "Samuel Colvin"\n}'

In [29]:
import json

data = json.loads(resp.choices[0].message.content)

In [30]:
data

{'name': 'pydantic', 'nickname': 'pydantic', 'author': 'Samuel Colvin'}

In [31]:
package = PythonPackage(**data)

print(package.name)   # "pydantic"
#print(package.nickname)   # "pydantic"
print(package.author) # "Samuel Colvin"

pydantic
Samuel Colvin


In [32]:
resp.choices[0].message.content

'{\n    "name": "pydantic",\n    "nickname": "pydantic",\n    "author": "Samuel Colvin"\n}'

**Example**

In [33]:
from pydantic import BaseModel, Field

In [34]:
class PatientInfo(BaseModel):
    name: str      = Field(..., 
                           title="Patient's full name",
                           description="First name then last name"
                          )
    age: int       = Field(..., 
                           ge=0, 
                           le=130, 
                           title="Age of the patient",
                           description="Age must be between 0 and 130")
    diagnosis: str = Field(..., 
                           title="Recent diagnosis",
                           description="Medical diagnosis")

🔹 Field(...) gives metadata, constraints, or defaults.

🔹 Used for schema validation in LLM output parsing.

In [35]:
# Example
data = {
    "name": "Aarav Sharma",
    "age": 45,
    "diagnosis": "Hypertension"
}

In [36]:
patient = PatientInfo(**data)
print(patient)

name='Aarav Sharma' age=45 diagnosis='Hypertension'


#### Nested Models

Use when LLM returns structured nested outputs:

In [37]:
class Medication(BaseModel):
    name: str
    dose: str

class PatientRecord(BaseModel):
    patient: PatientInfo
    medications: list[Medication]

In [38]:
# LLM output
llm_data = {
    "patient": {"name": "John", "age": 40, "diagnosis": "TNBC"},
    "medications": [
        {"name": "Carboplatin", "dose": "150mg"},
        {"name": "Paclitaxel", "dose": "100mg"}
    ]
}

In [39]:
record = PatientRecord(**llm_data)
print(record)

patient=PatientInfo(name='John', age=40, diagnosis='TNBC') medications=[Medication(name='Carboplatin', dose='150mg'), Medication(name='Paclitaxel', dose='100mg')]


### Constraints, Defaults, Optional Fields

Use Case: Patient Registration Data

In [40]:
from pydantic import BaseModel, Field, ValidationError, constr, conint
from typing import Optional
from pprint import pprint

In [41]:
class PatientRegistration(BaseModel):
    # Required field with constraints: must be alphabetic, min 2 chars
    name: constr(min_length=2, pattern="^[A-Za-z ]+$") = Field(..., description="Full name of the patient")

    # Age must be between 0 and 120
    age: conint(ge=0, le=120) = Field(..., description="Age in years")

    # Optional email, with default = None
    email: Optional[str] = Field(None, description="Optional email address")

    # Optional country, with default value
    country: str = Field(default="India", description="Country of residence")

    # Consent checkbox (True/False), default is False
    consent_given: bool = Field(default=False, description="Has the patient given consent?")


In [42]:
# Test case with complete valid data
try:
    patient = PatientRegistration(
        name="Amit Sharma",
        age=34,
        email="amit@example.com",
        consent_given=True
    )
    print("\n✅ Valid Patient Registration:")
    pprint(patient.model_dump(), indent=2)

except ValidationError as e:
    print("\n❌ Validation Errors:")
    pprint(e.errors(), indent=2)


✅ Valid Patient Registration:
{ 'age': 34,
  'consent_given': True,
  'country': 'India',
  'email': 'amit@example.com',
  'name': 'Amit Sharma'}


In [43]:
try:
    patient = PatientRegistration(name="A1", age=130)
    
    print("\n✅ Valid Patient Registration:")
    pprint(patient.model_dump(), indent=2)

except ValidationError as e:
    print("\n❌ Validation Errors:")
    pprint(e.errors(), indent=2)


❌ Validation Errors:
[ { 'ctx': {'pattern': '^[A-Za-z ]+$'},
    'input': 'A1',
    'loc': ('name',),
    'msg': "String should match pattern '^[A-Za-z ]+$'",
    'type': 'string_pattern_mismatch',
    'url': 'https://errors.pydantic.dev/2.12/v/string_pattern_mismatch'},
  { 'ctx': {'le': 120},
    'input': 130,
    'loc': ('age',),
    'msg': 'Input should be less than or equal to 120',
    'type': 'less_than_equal',
    'url': 'https://errors.pydantic.dev/2.12/v/less_than_equal'}]


**Example**

In [44]:
# Import Pydantic for data validation
from pydantic import BaseModel, field_validator

In [45]:
# Optional types and regex support
from typing import Optional
import re

In [46]:
# To parse timestamp strings into datetime objects
from datetime import datetime

In [47]:
# Define the schema using Pydantic
class LogEntry(BaseModel):
    timestamp: datetime            # The timestamp of the log entry
    process_id: int                # The numeric process ID
    log_level: str                 # Log level: Note, Warning, or ERROR
    message: str                   # The actual log message

    # Custom validator to ensure only allowed log levels
    # before accepting the value for log_level, run it through this custom validation function
    # It applies only to the log_level field, and allows you to enforce rules 
    # beyond just type checking.
    @field_validator('log_level')
    def valid_level(cls, v):   # Reference to the Pydantic model class (LogEntry), value passed to the log_level field
        allowed = ['Note', 'Warning', 'ERROR']
        
        if v not in allowed:
            raise ValueError(f'Invalid log level: {v}')  # Raise error if level is not allowed
        return v

In [48]:
# Define regex pattern to extract components from log line
log_pattern = re.compile(
    r'^(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) '  # Captures datetime
    r'(?P<process_id>\d+) '                                 # Captures process ID
    r'\[(?P<log_level>\w+)\] '                              # Captures log level inside brackets
    r'(?P<message>.+)$'                                     # Captures the full message
)

In [ ]:
# Set path to the log file (you mentioned this is your actual log file path)
log_file_path = r"D:\AI-DATASETS\02-PROJECT-DATA\deepak-log-files\log_file_1.txt.txt"

In [ ]:
# Prepare a list to hold all valid parsed log entries
parsed_logs = []

In [ ]:
# Open the file for reading
with open(log_file_path, 'r') as f:
    for line in f:  # Process each line one by one
        match = log_pattern.match(line.strip())  # Apply regex to extract fields
        if match:
            try:
                # Try to create a validated LogEntry object
                entry = LogEntry(**match.groupdict())  # Convert regex match dict into model
                parsed_logs.append(entry)             # Store the valid entry
            except Exception as e:
                # Print error if validation fails
                print(f"Validation error: {e} on line: {line.strip()}")
        else:
            # If regex doesn't match, it's an unrecognized log format
            print(f"Unmatched line: {line.strip()}")

Validation error: 1 validation error for LogEntry
log_level
  Value error, Invalid log level: XYZ [type=value_error, input_value='XYZ', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error on line: 2025-03-27 08:13:47 0 [XYZ] InnoDB: Progress in purging redo log: 80% complete


In [ ]:
# Print summary after parsing
print(f"\nTotal valid log entries: {len(parsed_logs)}")


Total valid log entries: 183


In [ ]:
type(parsed_logs)

list

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', 400)

# Load valid parsed logs into a DataFrame
df_logs = pd.DataFrame([log.dict() for log in parsed_logs])

C:\Users\bhupe\AppData\Local\Temp\ipykernel_25876\3880585186.py:5: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  df_logs = pd.DataFrame([log.dict() for log in parsed_logs])


In [ ]:
df_logs.sample(10)

,timestamp,process_id,log_level,message
146,2025-03-27 09:16:00,143,ERROR,Fatal error: Can't open the MySQL server
86,2025-03-27 08:46:00,83,ERROR,Duplicate entry '12345' for key 'PRIMARY'
174,2025-03-27 09:30:00,171,ERROR,Access denied for user 'root'@'localhost' (using password: YES)
68,2025-03-27 08:37:00,65,ERROR,Incorrect column count during row insert
147,2025-03-27 09:16:30,144,ERROR,Failed to start MySQL server because of SSL certificate issue
180,2025-03-27 09:33:00,177,ERROR,Table 'mysql.tables_priv' doesn't exist
172,2025-03-27 09:29:00,169,ERROR,Table 'mysql.db' has no PRIMARY KEY
118,2025-03-27 09:02:00,115,ERROR,Cannot add foreign key constraint
70,2025-03-27 08:38:00,67,ERROR,Could not allocate a new connection handle
143,2025-03-27 09:14:30,140,ERROR,"SQLSTATE[22001]: Data truncation: string data, right truncated: 'index_column'"


some more validations ...

In [ ]:
from pydantic import Field, model_validator

In [ ]:
class LogEntry(BaseModel):
    timestamp: datetime = Field(description="The timestamp of the log entry")
    process_id: int     = Field(ge=0, description="The numeric process ID, must be >= 0")
    log_level: str      = Field(description="Log level: Note, Warning, or ERROR")
    message: str        = Field(min_length=5, max_length=500, description="The actual log message")

    # Log level validation 
    @field_validator('log_level')
    def validate_log_level(cls, v):
        allowed = ['Note', 'Warning', 'ERROR']
        normalized = v.strip().title() if v.lower() != "error" else "ERROR"  # Normalize
        if normalized not in allowed:
            raise ValueError(f'Invalid log level: {v}')
        return normalized

    # Timestamp validation 
    @field_validator('timestamp')
    def validate_timestamp(cls, v):
        now = datetime.now()
        if v > now:
            raise ValueError("Timestamp cannot be in the future")
        if v.year < 2000:
            raise ValueError("Timestamp is too far in the past")
        return v

    # Process ID validation 
    @field_validator('process_id')
    def validate_pid(cls, v):
        if v > 65535:
            raise ValueError("Process ID is too high")
        return v

    # Message content validation 
    @field_validator('message')
    def validate_message(cls, v):
        if not v.strip():
            raise ValueError("Message cannot be empty or whitespace")
        return v

    # Cross-field validation: e.g., certain content must exist for errors 
    # @model_validator(mode='after')
    # def check_error_message(self):
    #     if self.log_level == "ERROR" and not any(keyword in self.message.lower() for keyword in ["fail", "error", "crash", "lost", "unable", "denied"]):
    #         raise ValueError("ERROR logs should mention specific failure-related keywords")
    #     return self

In [ ]:
# Prepare a list to hold all valid parsed log entries
parsed_logs = []

In [ ]:
# Open the file for reading
with open(log_file_path, 'r') as f:
    for line in f:  # Process each line one by one
        match = log_pattern.match(line.strip())  # Apply regex to extract fields
        if match:
            try:
                # Try to create a validated LogEntry object
                entry = LogEntry(**match.groupdict())  # Convert regex match dict into model
                parsed_logs.append(entry)             # Store the valid entry
            except Exception as e:
                # Print error if validation fails
                print(f"Validation error: {e} on line: {line.strip()}")
        else:
            # If regex doesn't match, it's an unrecognized log format
            print(f"Unmatched line: {line.strip()}")

Validation error: 1 validation error for LogEntry
log_level
  Value error, Invalid log level: XYZ [type=value_error, input_value='XYZ', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error on line: 2025-03-27 08:13:47 0 [XYZ] InnoDB: Progress in purging redo log: 80% complete


In [ ]:
# Load valid parsed logs into a DataFrame
df_logs = pd.DataFrame([log.dict() for log in parsed_logs])
df_logs.shape

C:\Users\bhupe\AppData\Local\Temp\ipykernel_25876\2727939171.py:2: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  df_logs = pd.DataFrame([log.dict() for log in parsed_logs])


(183, 4)

#### Handling unstructured inputs from LLMs

**Example Use Case: Extracting Patient Info**

In [ ]:
# Suppose the LLM returns this unstructured response
llm_output = """
Name: John Doe
Age: 45
Symptoms: fever, cough
Diagnosis: Likely flu
"""

Define Pydantic model

In [ ]:
from pydantic import BaseModel, Field, ValidationError
from typing import List, Optional

In [ ]:
class PatientInfo(BaseModel):
    name: str
    age: int
    symptoms: List[str]
    diagnosis: Optional[str] = None

Parse the unstructured text using regex or fuzzy matching

In [ ]:
import re

def extract_patient_info(text: str) -> dict:
    return {
        "name": re.search(r"Name:\s*(.*)", text).group(1),
        "age": int(re.search(r"Age:\s*(\d+)", text).group(1)),
        "symptoms": [s.strip() for s in re.search(r"Symptoms:\s*(.*)", text).group(1).split(',')],
        "diagnosis": re.search(r"Diagnosis:\s*(.*)", text).group(1),
    }

 Validate and structure using Pydantic

In [ ]:
try:
    patient_dict = extract_patient_info(llm_output)
    patient      = PatientInfo(**patient_dict)
    print(patient)
except ValidationError as e:
    print("Validation failed:", e)

name='John Doe' age=45 symptoms=['fever', 'cough'] diagnosis='Likely flu'


**What if the LLM returns JSON-like output?**

If the LLM is instructed to return JSON, and it nearly complies:

In [ ]:
import json

llm_json_output = """
{
    "name": "Alice",
    "age": "30",
    "symptoms": ["headache", "fatigue"],
    "diagnosis": null
}
"""

In [ ]:
try:
    parsed  = json.loads(llm_json_output)
    patient = PatientInfo(**parsed)
    print(patient)
except (json.JSONDecodeError, ValidationError) as e:
    print("Failed to parse or validate:", e)

name='Alice' age=30 symptoms=['headache', 'fatigue'] diagnosis=None


#### Nested models, constraints, error handling


Define Nested Models

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

In [ ]:
class Address(BaseModel):
    line1: str
    city: str
    zip: int

class PatientInfo(BaseModel):
    name: str
    age: int
    address: Address

Parse from dict

In [ ]:
parsed = {
    "name": "Anil Kapoor",
    "age": 1162,
    "address": {
        "line1": "123 Palm Street",
        "city": "Mumbai",
        "zip": 400001
    }
}

In [ ]:
patient = PatientInfo(**parsed)
print(patient)

name='Anil Kapoor' age=1162 address=Address(line1='123 Palm Street', city='Mumbai', zip=400001)


Field Constraints and Validation

In [ ]:
from pydantic import BaseModel, Field, conint, constr

In [ ]:
class PatientInfo(BaseModel):
    name: constr(min_length=2)
    age:  conint(ge=0, le=130)  # age between 0 and 130
    symptoms: list[str] = Field(min_items=1)

You’ll now get automatic errors if:

Name is too short

Age is negative

Symptoms list is empty



**Catch ValidationError**


In [ ]:
from pydantic import ValidationError

In [ ]:
# Here's your "bad" input to test error handling
bad_input_dict = {
    "name": "A",
    "age": -5,              # invalid: less than 0
    "symptoms": []          # invalid: empty list
}

In [ ]:
try:
    patient = PatientInfo(**bad_input_dict)
except ValidationError as e:
    print("❌ Validation failed")
    print(e.json(indent=2))

❌ Validation failed
[
  {
    "type": "string_too_short",
    "loc": [
      "name"
    ],
    "msg": "String should have at least 2 characters",
    "input": "A",
    "ctx": {
      "min_length": 2
    },
    "url": "https://errors.pydantic.dev/2.11/v/string_too_short"
  },
  {
    "type": "greater_than_equal",
    "loc": [
      "age"
    ],
    "msg": "Input should be greater than or equal to 0",
    "input": -5,
    "ctx": {
      "ge": 0
    },
    "url": "https://errors.pydantic.dev/2.11/v/greater_than_equal"
  },
  {
    "type": "too_short",
    "loc": [
      "symptoms"
    ],
    "msg": "List should have at least 1 item after validation, not 0",
    "input": [],
    "ctx": {
      "field_type": "List",
      "min_length": 1,
      "actual_length": 0
    },
    "url": "https://errors.pydantic.dev/2.11/v/too_short"
  }
]
